# Chapter 11 Tutorial - Part 1: Data Analytics & Warehousing

Welcome to the first half of the Chapter 11 tutorial. This notebook is designed to take approximately 60 minutes to complete. 

## Section 1: Introduction to Data Analytics
Data analytics is the processing of data to infer patterns, correlations, or models for prediction. Businesses rely on these insights to make critical decisions at two primary levels:
1. **Individual Level**: Tailoring experiences, such as suggesting a specific product to a specific user.
2. **Global Level**: Making macro decisions, such as determining overall manufacturing quantities or inventory stocking levels.

### Exercise 1: Rule-Based Business Decisions
Before we get into complex machine learning (which we will cover in Part 2), many business decisions are made using simple descriptive patterns and rules derived from data.

In the cell below, we will create a base dataset for a retail company. Your task is to implement a simple "Decision Rule" that identifies which customers should receive a special re-engagement advertisement based on their spending history.

In [5]:
import pandas as pd
import numpy as np

# Sample customer database
customers = {
    'customer_id': [101, 102, 103, 104, 105],
    'age': [25, 45, 34, 50, 23],
    'annual_spend': [500, 15000, 2000, 12000, 800],
    'last_purchase_days_ago': [10, 150, 5, 200, 2]
}
df_cust = pd.DataFrame(customers)

# Task Completion 
high_spenders = df_cust['annual_spend'] > 5000
inactive_customers = df_cust['last_purchase_days_ago'] > 30
df_cust['send_ad'] = high_spenders & inactive_customers

print("Targeting Analysis Results:")
print(df_cust)

Targeting Analysis Results:
   customer_id  age  annual_spend  last_purchase_days_ago  send_ad
0          101   25           500                      10    False
1          102   45         15000                     150     True
2          103   34          2000                       5    False
3          104   50         12000                     200     True
4          105   23           800                       2    False


---
## Section 2: Data Gathering and ETL/ELT Processes
Raw data is rarely ready for analysis immediately. It must be **Extracted** from source systems, **Transformed** to fit a common schema (and cleaned), and **Loaded** into the data warehouse. This is known as the ETL process.

### Exercise 2: Data Cleansing
Data transformation heavily involves "cleansing"—correcting mistakes like zip code errors, standardizing text casing, or removing stray whitespace and punctuation. 

Complete the code below to clean a messy extraction from a regional sales database.

In [6]:
# Messy source data extracted from a legacy system
source_data = {
    'store_id': [1, 2, 3],
    'city': ['New York', 'Los Angeles ', 'new york'], # Inconsistent casing and whitespace
    'revenue': ['1,200', '2500', '1,100'] # Stored as strings with commas
}
df_source = pd.DataFrame(source_data)

# Task completion
# Cleaning city
df_source['city'] = df_source['city'].str.strip().str.lower()

# Cleaning revenue
df_source['revenue'] = df_source['revenue'].str.replace(',', '').astype(float)

print("Cleaned Data Ready for Warehouse Loading:")
print(df_source)

Cleaned Data Ready for Warehouse Loading:
   store_id         city  revenue
0         1     new york   1200.0
1         2  los angeles   2500.0
2         3     new york   1100.0


---
## Section 3: Data Warehouse Architectures
Corporate decision-making requires a unified view of historical data. Data warehouses shift query loads away from the Online Transaction Processing (OLTP) systems to avoid slowing down daily operations.

There are two main gathering architectures:
1. **Source-driven**: Data sources automatically transmit new information to the warehouse periodically (e.g., nightly batches).
2. **Destination-driven**: The data warehouse actively requests and pulls new information from the data sources.

### Exercise 3: Simulating a Destination-Driven Pull
We need to simulate a warehouse pulling data. To avoid duplicates, the warehouse should only load records that do not already exist in its repository.

In [9]:
# Current IDs already stored in the Data Warehouse
warehouse_existing_ids = [101, 102]

# Fresh data pulled from the transactional database
df_new_extract = pd.DataFrame({
    'transaction_id': [101, 102, 103, 104],
    'amount': [15.50, 22.00, 9.99, 54.20]
})

# TODO: Filter df_new_extract to keep ONLY the rows where 'transaction_id' 
# is NOT present in the warehouse_existing_ids list.
# Hint: Use the ~ operator with .isin()
already_exists = df_new_extract['transaction_id'].isin(warehouse_existing_ids)
is_new = ~already_exists #take vice versa

df_incremental_load = df_new_extract[is_new]

print("Incremental Load (Only New Records):")
print(df_incremental_load)

Incremental Load (Only New Records):
   transaction_id  amount
2             103    9.99
3             104   54.20


---
## Section 4: Multidimensional Data Modeling
Once data is in the warehouse, it is typically structured into specific types of tables to make analytical querying efficient:
* **Fact Tables**: These are extremely large tables containing the "events" of the business. They hold **Measure Attributes** (numerical values you can aggregate, like quantity or price) and foreign keys.
* **Dimension Tables**: These are smaller, reference tables. They hold **Dimension Attributes** (descriptive info like store location, item category, or time/date details).

### Exercise 4: Defining Facts and Dimensions
Let's create the components of a multidimensional model.

In [10]:
# Task completion
import pandas as pd

df_sales_fact = pd.DataFrame({
    'item_id':   [1,     2,      3,      1   ],  # links to df_item_dim
    'store_id':  [101,   101,    102,    102 ],  # links to df_store_dim
    'quantity':  [2,     1,      4,      3   ],  # measurable numbers
    'price':     [9.99,  24.99,  4.99,   9.99]  # measurable numbers
})


df_item_dim = pd.DataFrame({
    'item_id':   [1,          2,          3       ],
    'item_name': ['Notebook', 'Headphones','Pen'  ],
    'category':  ['Stationery','Electronics','Stationery']
})

# ── DIMENSION TABLE: Stores ───────────────────────────────────────
# store_id here must match the store_ids used above in df_sales_fact.
df_store_dim = pd.DataFrame({
    'store_id': [101,        102         ],
    'city':     ['New York', 'Austin'    ],
    'state':    ['NY',       'TX'        ]
})

print("Fact and Dimension Tables initialized successfully.")
print("\n--- Sales Fact Table ---")
print(df_sales_fact)
print("\n--- Item Dimension Table ---")
print(df_item_dim)
print("\n--- Store Dimension Table ---")
print(df_store_dim)

Fact and Dimension Tables initialized successfully.

--- Sales Fact Table ---
   item_id  store_id  quantity  price
0        1       101         2   9.99
1        2       101         1  24.99
2        3       102         4   4.99
3        1       102         3   9.99

--- Item Dimension Table ---
   item_id   item_name     category
0        1    Notebook   Stationery
1        2  Headphones  Electronics
2        3         Pen   Stationery

--- Store Dimension Table ---
   store_id      city state
0       101  New York    NY
1       102    Austin    TX


---
## Section 5: The Star Schema
The most common warehouse schema is the **Star Schema**. It gets its name because the large Fact Table sits in the center, with the smaller Dimension Tables radiating outward like the points of a star.

To analyze data, we must join the central fact table with its dimension tables to get a complete, human-readable view.

### Exercise 5: Building the Star Schema View
Combine the fact and dimension tables you created in the previous step.

In [11]:
# If you did not complete Exercise 4, use this backup data by uncommenting:
df_sales_fact = pd.DataFrame({'item_id': [1, 2], 'store_id': [10, 11], 'qty': [5, 2]})
df_item_dim = pd.DataFrame({'item_id': [1, 2], 'item_name': ['Laptop', 'Mouse']})
df_store_dim = pd.DataFrame({'store_id': [10, 11], 'city': ['Tel Aviv', 'Haifa']})


df_step_1 = pd.merge(
    df_sales_fact, 
    df_item_dim,   
    on='item_id',   
    how='left'     
)
df_star_schema = pd.merge(
    df_step_1,   
    df_store_dim, 
    on='store_id',  
    how='left'
)

print("Fully Joined Star Schema Data:")
print(df_star_schema)

Fully Joined Star Schema Data:
   item_id  store_id  qty item_name      city
0        1        10    5    Laptop  Tel Aviv
1        2        11    2     Mouse     Haifa


---
## Section 6: Storage Optimization (Column-Oriented Storage)
Relational databases typically store data row-by-row. Data warehouses, however, often use **column-oriented storage**. 

Instead of storing a full record together, all values for `item_id` are stored as one compressed array, all values for `price` as another array, etc. This drastically reduces I/O costs because analytical queries usually only need a few specific columns, not the whole row.

### Exercise 6: Simulating Columnar vs Row Operations
In pandas, vector operations on a single Series (column) simulate the efficiency of a columnar database.

In [12]:
# Creating a large simulated fact table
large_fact_table = pd.DataFrame({
    'id': range(100000),
    'price': np.random.rand(100000) * 100,
    'tax': np.random.rand(100000) * 10
})

# Row-oriented approach (Iterating through rows - VERY SLOW)
def row_oriented_sum(df):
    total = 0
    for index, row in df.iterrows():
        total += row['price']
    return total

# Task Completion
def column_oriented_sum(df):
    return df['price'].sum()

import time

start = time.time()
row_oriented_sum(large_fact_table)
print(f"Row operation took: {time.time() - start:.4f} seconds")

start = time.time()
column_oriented_sum(large_fact_table)
print(f"Column operation took: {time.time() - start:.4f} seconds")

Row operation took: 2.8958 seconds
Column operation took: 0.0037 seconds


---
## Section 7: Online Analytical Processing (OLAP) & Pivot Tables
OLAP allows users to interactively analyze and summarize data. The foundational tool for this is the **Cross-Tabulation**, widely known as a **Pivot Table**.

In a pivot table:
* One dimension attribute forms the rows.
* Another dimension attribute forms the columns.
* The cells contain aggregated measure attributes (like the sum of quantity).

### Exercise 7: Creating a Pivot Table
We will set up a master OLAP dataset and create a cross-tab.

In [13]:
# Master OLAP dataset (simplified star schema view)
olap_data = {
    'item': ['dress', 'dress', 'pants', 'pants', 'shirt', 'shirt', 'skirt', 'skirt'],
    'color': ['dark', 'pastel', 'dark', 'pastel', 'white', 'dark', 'pastel', 'white'],
    'size': ['small', 'small', 'medium', 'large', 'small', 'medium', 'large', 'small'],
    'qty': [2, 4, 6, 1, 17, 6, 15, 2]
}
df_olap = pd.DataFrame(olap_data)

pivot_table = pd.pivot_table(
    df_olap,
    index='item',       # rows
    columns='color',    # columns
    values='qty',       # what to measure
    aggfunc='sum',      # how to combine multiple matches
    fill_value=0        # replace empty cells with 0
)

print("OLAP Pivot Table (Item by Color):")
print(pivot_table)

OLAP Pivot Table (Item by Color):
color  dark  pastel  white
item                      
dress     2       4      0
pants     6       1      0
shirt     6       0     17
skirt     0      15      2


---
## Section 8: OLAP Operations - Slicing
The data cube is a multidimensional generalization of a cross-tab. 

**Slicing** is the operation of creating a cross-tab for fixed values only. For example, slicing the data cube to only look at sales where the color is exactly 'dark'.

### Exercise 8: Slicing the Data Cube
Filter the master `df_olap` dataset to perform a slice.

In [14]:
# Task Completion
df_slice = df_olap[df_olap['color'] == 'dark']


print("Sliced Data Cube (Dark colors only):")
print(df_slice)

Sliced Data Cube (Dark colors only):
    item color    size  qty
0  dress  dark   small    2
2  pants  dark  medium    6
5  shirt  dark  medium    6


---
## Section 9: OLAP Operations - Dicing
**Dicing** is similar to slicing, but it restricts the data cube on multiple dimensions simultaneously. For example, fixing the color to 'dark' AND the size to 'medium'.

### Exercise 9: Dicing the Data Cube
Apply multiple filters to narrow down the dataset.

In [15]:
df_dice = df_olap[(df_olap['color'] == 'dark') & (df_olap['size'] == 'small')]
print("Diced Data Cube (Dark AND Medium):")
print(df_dice)

Diced Data Cube (Dark AND Medium):
    item color   size  qty
0  dress  dark  small    2


---
## Section 10: Hierarchies and Rollup
Dimensions often possess natural hierarchies (e.g., City -> State -> Country). 

**Rollup** is the operation of moving from finer-granularity data to coarser-granularity data. This involves aggregating away an attribute. **Drill-down** is the exact opposite operation.

### Exercise 10: Manual Rollup
We will manually calculate the rollup totals by stepping up the hierarchy.

In [16]:
# Base level of detail (Grouping by both Item and Color)
detail_level = df_olap.groupby(['item', 'color'])['qty'].sum().reset_index()

rollup_item = df_olap.groupby('item')['qty'].sum().reset_index()

grand_total = df_olap['qty'].sum()

print("Detail Level:\n", detail_level)
print("\nRollup (Item Level):\n", rollup_item)
print("\nGrand Total:\n", grand_total)

Detail Level:
     item   color  qty
0  dress    dark    2
1  dress  pastel    4
2  pants    dark    6
3  pants  pastel    1
4  shirt    dark    6
5  shirt   white   17
6  skirt  pastel   15
7  skirt   white    2

Rollup (Item Level):
     item  qty
0  dress    6
1  pants    7
2  shirt   23
3  skirt   17

Grand Total:
 53


---
## Section 11: The SQL CUBE Operation
In SQL, the `CUBE` operation automatically computes the union of `GROUP BY` operations on *every possible subset* of the specified attributes.

For two dimensions (item and color), a `CUBE` operation computes 4 groupings:
1. Group by (item, color)
2. Group by (item)
3. Group by (color)
4. Group by ( ) -> Grand total

### Exercise 11: Simulating the CUBE Operation in Pandas
Pandas doesn't have a direct `.cube()` method, so we must concatenate multiple group-bys to simulate what a database engine does under the hood.

In [ ]:
# 1. Group by both (item, color)
g1 = df_olap.groupby(['item', 'color'])['qty'].sum().reset_index()

# 2. Group by item only
g2 = df_olap.groupby(['item'])['qty'].sum().reset_index()

# 3. Group by color only
g3 = df_olap.groupby(['color'])['qty'].sum().reset_index()

# 4. Grand total
g4 = pd.DataFrame({'qty': [df_olap['qty'].sum()]})

# Task completion
df_cube = pd.concat([g1, g2, g3, g4], ignore_index=True).fillna('ALL')

print("Simulated SQL CUBE(item, color) Results:")
print(df_cube)

Simulated SQL CUBE(item, color) Results:
     item   color  qty
0   dress    dark    2
1   dress  pastel    4
2   pants    dark    6
3   pants  pastel    1
4   shirt    dark    6
5   shirt   white   17
6   skirt  pastel   15
7   skirt   white    2
8   dress     ALL    6
9   pants     ALL    7
10  shirt     ALL   23
11  skirt     ALL   17
12    ALL    dark   14
13    ALL  pastel   20
14    ALL   white   19
15    ALL     ALL   53


: 

---
**End of Part 1.**

You have successfully covered the foundational engineering concepts of Data Analytics, from ETL cleansing and Star Schemas to complex OLAP data cube operations. 

Once you have completed the exercises above, you are ready to proceed to **Part 2: Data Mining**.